In [106]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import numpy as np

In [107]:
bus = pd.read_csv('Nodes/Bus.csv')
fgc = pd.read_csv('Nodes/FGC.csv')
metro = pd.read_csv('Nodes/Metro.csv')
tram = pd.read_csv('Nodes/Tram.csv')

all_stops = pd.concat([ bus, fgc, metro, tram], ignore_index=True)

exchange_edges = pd.read_csv('Edges/Exchanges.csv')

In [108]:
exchange_edges = exchange_edges.merge(all_stops[['id','stop_id']], left_on='dest', right_on='id', how='left')

# Tram

In [109]:
exchange_edges_tram = exchange_edges[exchange_edges['dest'].str.startswith('T')]

In [110]:
stop_times_x = pd.read_table('Data/Tram/TBX/stop_times.txt', sep=',')
stop_times_S = pd.read_table('Data/Tram/TBS/stop_times.txt', sep=',')
stop_times = pd.concat([stop_times_x, stop_times_S])    
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[['trip_id','stop_id','departure_time']]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) < 11]

In [111]:
stop_times

,trip_id,stop_id,departure_time
71,2555_0196,1127,07:01:00
72,2555_0001,1127,07:06:30
73,2555_0001,1126,07:08:10
74,2555_0001,1125,07:10:10
75,2555_0001,1122,07:12:10
...,...,...,...
10511,1910_0451,2017,10:47:50
10512,1910_0451,2019,10:49:40
10513,1910_0452,2019,10:55:00
10514,1910_0452,2117,10:57:10


In [112]:
stops_x = pd.read_table('Data/Tram/TBX/stops.txt', sep=',')[['stop_id','stop_name','stop_desc']]
stops_s = pd.read_table('Data/Tram/TBS/stops.txt', sep=',')[['stop_id','stop_name','stop_desc']]
stops = pd.concat([stops_x, stops_s])
stops = stops[~stops['stop_id'].str.startswith('S')]
stops['stop_id'] = stops['stop_id'].astype(int)

In [113]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]

In [114]:
trips_x = pd.read_table('Data/Tram/TBX/trips.txt', sep=',')
trips_s = pd.read_table('Data/Tram/TBS/trips.txt', sep=',')
trips = pd.concat([trips_x, trips_s])
trips = trips[['route_id','trip_id']]

In [115]:
trips

,route_id,trip_id
0,2,2555_0194
1,2,2555_0195
2,2,2555_0196
3,2,2555_0001
4,3,2555_0362
...,...,...
1674,5,1910_0509
1675,5,1910_0510
1676,6,1910_0511
1677,6,1910_0512


In [116]:
stops_w_times

,stop_id,stop_name,stop_desc,trip_id,departure_time
0,1032,Sant Feliu|C.C.,A_CCML,2555_0362,08:25:00
1,1032,Sant Feliu|C.C.,A_CCML,2555_0363,08:28:00
2,1032,Sant Feliu|C.C.,A_CCML,2555_0507,08:37:00
3,1032,Sant Feliu|C.C.,A_CCML,2555_0508,08:41:20
4,1032,Sant Feliu|C.C.,A_CCML,2555_0394,07:37:00
...,...,...,...,...,...
10015,2108,El Maresme,T_MRSM,1910_0189,10:05:00
10016,2108,El Maresme,T_MRSM,1910_0190,10:24:30
10017,2108,El Maresme,T_MRSM,1910_0191,10:30:00
10018,2108,El Maresme,T_MRSM,1910_0192,10:48:30


In [117]:
stops_w_times

,stop_id,stop_name,stop_desc,trip_id,departure_time
0,1032,Sant Feliu|C.C.,A_CCML,2555_0362,08:25:00
1,1032,Sant Feliu|C.C.,A_CCML,2555_0363,08:28:00
2,1032,Sant Feliu|C.C.,A_CCML,2555_0507,08:37:00
3,1032,Sant Feliu|C.C.,A_CCML,2555_0508,08:41:20
4,1032,Sant Feliu|C.C.,A_CCML,2555_0394,07:37:00
...,...,...,...,...,...
10015,2108,El Maresme,T_MRSM,1910_0189,10:05:00
10016,2108,El Maresme,T_MRSM,1910_0190,10:24:30
10017,2108,El Maresme,T_MRSM,1910_0191,10:30:00
10018,2108,El Maresme,T_MRSM,1910_0192,10:48:30


In [118]:
times_w_routes = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['stop_id','stop_name','route_id','departure_time'], keep='first', inplace=True)
times_w_routes['route_id'] = 'T' + times_w_routes['route_id'].astype(int).astype(str)
times_w_routes['stop_desc'] = times_w_routes['stop_desc'].str[2:]
times_w_routes['stop_desc'] = times_w_routes['stop_desc'].replace({'RIGL':'SRMN','LLEV':'BDOR','MRSM':'DGMR','CATA':'CTLN','JOAN':'STJB','MRTI':'STMR','SROC':'STRC'})

In [119]:
times_w_routes

,stop_id,stop_name,stop_desc,trip_id,departure_time,route_id
0,1032,Sant Feliu|C.C.,CCML,2555_0362,08:25:00,T3
1,1032,Sant Feliu|C.C.,CCML,2555_0363,08:28:00,T3
2,1032,Sant Feliu|C.C.,CCML,2555_0507,08:37:00,T3
3,1032,Sant Feliu|C.C.,CCML,2555_0508,08:41:20,T3
4,1032,Sant Feliu|C.C.,CCML,2555_0394,07:37:00,T3
...,...,...,...,...,...,...
9920,2108,El Maresme,DGMR,1908_0194,10:00:30,T4
9921,2108,El Maresme,DGMR,1908_0195,10:05:00,T4
9923,2108,El Maresme,DGMR,1908_0197,10:30:00,T4
9924,2108,El Maresme,DGMR,1908_0198,10:48:30,T4


In [120]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id', 'departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )
avg_wait_times = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .assign(wait_time=lambda df: df['interval_minutes'] / 2)
    .groupby(['stop_id', 'route_id', 'stop_name', 'stop_desc'], as_index=False)['wait_time']
    .mean()
 )
avg_wait_times = avg_wait_times[['stop_desc', 'route_id', 'wait_time']]
avg_wait_times['id'] = 'T' + '-' + avg_wait_times['route_id'] + '-' + avg_wait_times['stop_desc']
avg_wait_times = avg_wait_times[['id', 'wait_time']]

In [121]:
exchange_edges_tram = exchange_edges_tram.merge(avg_wait_times, left_on='dest', right_on='id', how='left')
exchange_edges_tram = exchange_edges_tram[['origen', 'dest', 'tram', 'mode','lines','type','time',
                                           'wait_time','directed','geometry']]


# FCG

In [122]:
exchange_edges_fgc = exchange_edges[exchange_edges['dest'].str.startswith('F')]

In [123]:
stop_times = pd.read_table('Data/FGC/gtfs_fgc/stop_times.txt', sep=',')
stop_times = stop_times[['trip_id','stop_id','departure_time']]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) < 11]

In [124]:
stops = pd.read_table('Data/FGC/gtfs_fgc/stops.txt', sep=',')[['stop_id','stop_name']]

In [125]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]

In [126]:
trips = pd.read_table('Data/FGC/gtfs_fgc/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
trips = trips[['route_id','trip_id','trip_headsign']]
valid = ['L6', 'L7', 'L8', 'L12','S1']
trips = trips[trips['route_id'].isin(valid)]

In [127]:
times_w_routes = trips.merge(stops_w_times, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['route_id','stop_name','departure_time','trip_headsign'], keep='first', inplace=True)
times_w_routes = times_w_routes[times_w_routes['departure_time'].notna()]
times_w_routes['stop_id']  = times_w_routes['stop_id'].str[:2]

In [128]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id','trip_headsign', 'departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )

times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_wait_times = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .assign(wait_time=lambda df: df['interval_minutes'] / 2)
    .groupby(['route_id', 'stop_id', 'stop_name','trip_headsign'], as_index=False)['wait_time']
    .mean()
)
avg_wait_times = avg_wait_times.groupby(['route_id', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times['stop_id'] = 'M' + '-' + avg_wait_times['route_id'] + '-' + avg_wait_times['stop_id']
avg_wait_times

,route_id,stop_id,stop_name,wait_time
0,L12,M-L12-RE,Reina Elisenda,1.602361
1,L12,M-L12-SR,Sarrià,1.602361
2,L6,M-L6-BN,La Bonanova,1.667934
3,L6,M-L6-GR,Gràcia,1.434656
4,L6,M-L6-MN,Muntaner,1.669772
5,L6,M-L6-PC,Barcelona - Plaça Catalunya,1.435007
6,L6,M-L6-PR,Provença,1.408198
7,L6,M-L6-SG,Sant Gervasi,1.665641
8,L6,M-L6-SR,Sarrià,1.698788
9,L6,M-L6-TT,Les Tres Torres,1.685035


In [129]:
exchange_edges_fgc = exchange_edges_fgc.merge(avg_wait_times[['stop_id','wait_time']], left_on='dest', right_on='stop_id', how='left')
exchange_edges_fgc = exchange_edges_fgc[['origen', 'dest', 'tram', 'mode','lines','type','time',
                                           'wait_time','directed','geometry']]

# Bus

In [130]:
exchange_edges_bus = exchange_edges[exchange_edges['dest'].str.startswith('B')]
exchange_edges_bus

,origen,dest,tram,mode,lines,type,time,directed,geometry,id,stop_id
0,SB-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - 59,Exchange - Self,1.000000,True,POINT (2.198984998801282 41.393104003754864),B-59-2,2
1,B-H16-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,H16 - 59,Exchange - Self,1.000000,True,POINT (2.198984998801282 41.393104003754864),B-59-2,2
2,B-V27-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,V27 - 59,Exchange - Self,1.000000,True,POINT (2.198984998801282 41.393104003754864),B-59-2,2
3,SB-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - H16,Exchange - Self,1.000000,True,POINT (2.198984998801282 41.393104003754864),B-H16-2,2
4,B-59-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,59 - H16,Exchange - Self,1.000000,True,POINT (2.198984998801282 41.393104003754864),B-H16-2,2
...,...,...,...,...,...,...,...,...,...,...,...
148204,B-B15-108088,B-B81-112736,Raval - Santa Rosa - Elcano - Roger de Llúria,Bus - Bus,B15 - B81,Exchange,4.400000,True,"LINESTRING (2.2135373 41.4439583, 2.2134839 41...",B-B81-112736,112736
148205,B-B81-109790,B-B81-112736,Escola Pere de Tera - Elcano - Roger de Llúria,Bus - Bus,B81 - B81,Exchange,3.650000,True,"LINESTRING (2.214254 41.4408375, 2.2142405 41....",B-B81-112736,112736
148206,B-B81-109968,B-B81-112736,CAP Santa Rosa - Elcano - Roger de Llúria,Bus - Bus,B81 - B81,Exchange,1.800000,True,"LINESTRING (2.214348 41.4412638, 2.2142369 41....",B-B81-112736,112736
148207,B-B15-110507,B-B81-112736,Raval - Centre Cívic - Elcano - Roger de Llúria,Bus - Bus,B15 - B81,Exchange,3.616667,True,"LINESTRING (2.2108499 41.44294, 2.2109213 41.4...",B-B81-112736,112736


## AMB

In [131]:
stop_times = pd.read_table('Data/GTFS_AMB/stop_times.txt', sep=',')
stop_times = stop_times[['trip_id', 'stop_id', 'departure_time']].dropna()

stop_times['departure_time'] = pd.to_datetime(
    stop_times['departure_time'],
    format='%H:%M:%S',
    errors='coerce'
)

stop_times = stop_times[stop_times['departure_time'].dt.hour.between(7, 11)]
stop_times

,trip_id,stop_id,departure_time
0,129.20.1.1.0,107229,1900-01-01 08:55:00
1,129.20.1.1.0,2234,1900-01-01 08:56:09
2,129.20.1.1.0,2235,1900-01-01 08:57:46
3,129.20.1.1.0,476,1900-01-01 08:58:45
4,129.20.1.1.0,365,1900-01-01 08:59:57
...,...,...,...
890865,516.8.2.4.23,109921,1900-01-01 11:53:39
890866,516.8.2.4.23,106867,1900-01-01 11:54:41
890867,516.8.2.4.23,106868,1900-01-01 11:55:16
890868,516.8.2.4.23,106869,1900-01-01 11:57:03


In [132]:
stops = pd.read_table('Data/GTFS_AMB/stops.txt', sep=',')[['stop_id','stop_name']]
stops

,stop_id,stop_name
0,109303,Eusebi Güell - Joaquim Auger
1,100005,Escola Busquets i Punset
2,109239,Faigs - Montseny
3,109244,Faigs - Freixe
4,1461,Av de Cornellà - Pont d'Esplugues
...,...,...
4922,100031,Palau Reial
4923,100032,Zona Universitària
4924,112179,Pl. Jacint Verdaguer
4925,112099,Dr. Robert - Av. Bufalà


In [133]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]
stops_w_times

,stop_id,stop_name,trip_id,departure_time
0,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.0,1900-01-01 09:17:44
1,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.1,1900-01-01 09:57:44
2,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.2,1900-01-01 10:37:44
3,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.3,1900-01-01 11:17:44
4,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.4,1900-01-01 11:57:44
...,...,...,...,...
246520,112100,Dr. Robert - Sardana,297.13.1.1.14,1900-01-01 10:55:48
246521,112100,Dr. Robert - Sardana,297.13.1.1.15,1900-01-01 11:10:48
246522,112100,Dr. Robert - Sardana,297.13.1.1.16,1900-01-01 11:25:48
246523,112100,Dr. Robert - Sardana,297.13.1.1.17,1900-01-01 11:40:48


In [134]:
trips = pd.read_table('Data/GTFS_AMB/trips.txt', sep=',')
trips = trips[['route_id','trip_id','trip_headsign']]
routes = pd.read_table('Data/GTFS_AMB/routes.txt', sep=',')[['route_id','route_short_name']] 
trips = trips.merge(routes, on='route_id', how='left')

In [135]:
stops_w_times

,stop_id,stop_name,trip_id,departure_time
0,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.0,1900-01-01 09:17:44
1,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.1,1900-01-01 09:57:44
2,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.2,1900-01-01 10:37:44
3,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.3,1900-01-01 11:17:44
4,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.4,1900-01-01 11:57:44
...,...,...,...,...
246520,112100,Dr. Robert - Sardana,297.13.1.1.14,1900-01-01 10:55:48
246521,112100,Dr. Robert - Sardana,297.13.1.1.15,1900-01-01 11:10:48
246522,112100,Dr. Robert - Sardana,297.13.1.1.16,1900-01-01 11:25:48
246523,112100,Dr. Robert - Sardana,297.13.1.1.17,1900-01-01 11:40:48


In [136]:
times_w_routes_amb = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes_amb = times_w_routes_amb[times_w_routes_amb['departure_time'].notna()]
times_w_routes_amb.drop_duplicates(subset=['route_id','stop_name','departure_time'], keep='first', inplace=True)
times_w_routes_amb['stop_id']  =  times_w_routes_amb['stop_id'].astype(int).astype(str)
times_w_routes_amb

,stop_id,stop_name,trip_id,departure_time,route_id,trip_headsign,route_short_name
0,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.0,1900-01-01 09:17:44,196,St. Boi L.,L74
1,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.1,1900-01-01 09:57:44,196,St. Boi L.,L74
2,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.2,1900-01-01 10:37:44,196,St. Boi L.,L74
3,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.3,1900-01-01 11:17:44,196,St. Boi L.,L74
4,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.4,1900-01-01 11:57:44,196,St. Boi L.,L74
...,...,...,...,...,...,...,...
244405,112100,Dr. Robert - Sardana,297.13.1.1.14,1900-01-01 10:55:48,297,Bonavista,B8
244406,112100,Dr. Robert - Sardana,297.13.1.1.15,1900-01-01 11:10:48,297,Bonavista,B8
244407,112100,Dr. Robert - Sardana,297.13.1.1.16,1900-01-01 11:25:48,297,Bonavista,B8
244408,112100,Dr. Robert - Sardana,297.13.1.1.17,1900-01-01 11:40:48,297,Bonavista,B8


## TMB

In [ ]:
stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) <= 11]
stop_times['stop_id'] = stop_times['stop_id'].astype(str)
stop_times = stop_times[stop_times['trip_id'].str.startswith('2')]
stop_times

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_40947/2739213128.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]


,trip_id,stop_id,departure_time
777221,2.1.121.3345302.4094,2.9678.700773,10:05:00
777222,2.1.121.3345302.4094,2.9680.700539,10:10:00
777223,2.1.121.3345302.4094,2.8319.700682,10:19:00
777224,2.1.121.3345302.4094,2.1846.700880,10:21:00
777225,2.1.121.3345302.4094,2.9937.700777,10:24:00
...,...,...,...
1828503,2.978.127.3379936.4099,2.2984.695617,23:14:00
1828504,2.978.127.3379936.4099,2.778.693751,23:16:00
1828509,2.978.127.3379936.4099,2.2532.689812,23:26:00
1828510,2.978.127.3379937.4099,2.2532.689812,23:26:00


In [151]:
stops = pd.read_table('Data/GTFS_TMB/stops.txt', sep=',')[['stop_id','stop_name']]
stops['stop_id'] = stops['stop_id'].astype(str)

In [152]:
stops_w_times = stop_times.merge(stops, on='stop_id', how='left')

In [153]:
trips = pd.read_table('Data/GTFS_TMB/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
routes = pd.read_table('Data/GTFS_TMB/routes.txt', sep=',')[['route_id','route_short_name']]
metro = ['L1','L2','L3','L4','L5','L9','L10','L11','L9N','L9S','L10N','L10S','FM','M1','M9','978']
trips = routes.merge(trips, on='route_id', how='left')
trips = trips[~trips['route_short_name'].isin(metro)]
trips

,route_id,route_short_name,trip_id,trip_headsign
40199,2.220.2999,D20,2.220.64.3369726.2999,Ernest Lluch
40200,2.220.2999,D20,2.220.64.3369728.2999,Ernest Lluch
40201,2.220.2999,D20,2.220.64.3369724.2999,Ernest Lluch
40202,2.220.2999,D20,2.220.64.3369708.2999,Ernest Lluch
40203,2.220.2999,D20,2.220.64.3369678.2999,Ernest Lluch
...,...,...,...,...
80906,2.196.2971,196,2.196.26.3064023.2971,Av. Tibidabo
80907,2.196.2971,196,2.196.26.3063966.2971,Av. Tibidabo
80908,2.196.2971,196,2.196.26.3064008.2971,Av. Tibidabo
80909,2.196.2971,196,2.196.26.3063968.2971,Av. Tibidabo


short routes, routes that don't represent real lines (ex trip Id = 2.239.105.3380438.4092)

In [154]:
times_w_routes_tmb = stops_w_times.merge(trips, on='trip_id', how='left')
#times_w_routes_tmb = times_w_routes_tmb[times_w_routes_tmb['departure_time'].notna()]
#times_w_routes_tmb.drop_duplicates(subset=['route_id','stop_name','departure_time'], keep='first', inplace=True)
times_w_routes_tmb['stop_id'] = times_w_routes_tmb['stop_id'].str.split('.').str[1]
times_w_routes_tmb

,trip_id,stop_id,departure_time,stop_name,route_id,route_short_name,trip_headsign
0,2.1.121.3345302.4094,9678,10:05:00,Florida,NaN,NaN,NaN
1,2.1.121.3345302.4094,9680,10:10:00,Torrassa,NaN,NaN,NaN
2,2.1.121.3345302.4094,8319,10:19:00,Santa Eulalia,NaN,NaN,NaN
3,2.1.121.3345302.4094,1846,10:21:00,Mercat Nou,NaN,NaN,NaN
4,2.1.121.3345302.4094,9937,10:24:00,Pl. Sants - Final de trajecte,NaN,NaN,NaN
...,...,...,...,...,...,...,...
241694,2.978.127.3379936.4099,2984,23:14:00,Cardenal Reig - Pisuerga,NaN,NaN,NaN
241695,2.978.127.3379936.4099,778,23:16:00,Av Sant Ramon Nonat - Cardenal Reig,NaN,NaN,NaN
241696,2.978.127.3379936.4099,2532,23:26:00,Carles III - Les Corts,NaN,NaN,NaN
241697,2.978.127.3379937.4099,2532,23:26:00,Carles III - Les Corts,NaN,NaN,NaN


In [155]:
times_w_routes_tmb['route_short_name'].unique()

array([nan, '6', '7', '13', '19', '21', '22', '23', '24', '27', '33',
       '34', '39', '46', '47', '52', '54', '55', '59', '60', '62', '63',
       '65', '67', '68', '70', '76', '78', '91', '94', '95', '96', '97',
       '102', '104', '107', '109', '111', '112', '113', '114', '115',
       '117', '118', '119', '120', '121', '122', '123', '124', '125',
       '126', '127', '128', '129', '130', '131', '132', '133', '134',
       '136', '137', '138', '141', '150', '157', '175', '180', '182',
       '183', '185', '191', '192', '196', 'V1', 'H2', 'V3', 'H4', 'V5',
       'H6', 'V7', 'H8', 'V9', 'H10', 'V11', 'H12', 'V13', 'H14', 'V15',
       'H16', 'V17', 'V19', 'D20', 'V21', 'V23', 'V25', 'V27', 'V29',
       'V31', 'V33', 'D40', 'X1', 'X2', 'X3', 'D50'], dtype=object)

In [161]:
times_w_routes_tmb[times_w_routes_tmb['route_short_name'] == '13'].groupby('trip_id')['stop_id'].nunique().sort_values(ascending=False)

trip_id
2.13.59.3070247.2833    5
2.13.59.3354332.2833    5
2.13.59.3354346.2833    5
2.13.59.3354345.2833    5
2.13.59.3354344.2833    5
                       ..
2.13.59.3070288.2833    5
2.13.59.3070287.2833    5
2.13.59.3070286.2833    5
2.13.59.3070285.2833    5
2.13.59.3354383.2833    5
Name: stop_id, Length: 135, dtype: int64

In [158]:
times_w_routes_tmb[times_w_routes_tmb['route_short_name'] == '6']['stop_id'].nunique()

8

In [ ]:
print(times_w_routes_tmb[times_w_routes_tmb['route_id'].isna()]['trip_id'].nunique())
print(times_w_routes_tmb[times_w_routes_tmb['route_id'].notna()]['trip_id'].nunique())

## Together

In [ ]:
times_w_routes = pd.concat([times_w_routes_tmb, times_w_routes_amb], ignore_index=True)
times_w_routes

In [ ]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id', 'trip_headsign','departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )
times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_wait_times = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .assign(wait_time=lambda df: df['interval_minutes'] / 2)
    .groupby(['route_id','route_short_name', 'stop_id', 'stop_name','trip_headsign'], as_index=False)['wait_time']
    .mean()
)

avg_wait_times = avg_wait_times.groupby(['route_id','route_short_name', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times['stop_id'] = 'B' + '-' + avg_wait_times['route_short_name'] + '-' + avg_wait_times['stop_id']
avg_wait_times

In [ ]:
exchange_edges_bus2 = exchange_edges_bus.merge(avg_wait_times[['stop_id','wait_time']], left_on='dest', right_on='stop_id', how='left')
exchange_edges_bus2 = exchange_edges_bus2[['origen', 'dest', 'tram', 'mode','lines','type','time',  'wait_time','directed','geometry']]
exchange_edges_bus2[exchange_edges_bus2['wait_time'].isna()]

In [ ]:
exchange_edges_bus2['line'] = exchange_edges_bus2['lines'].str.split(' - ').str[1]
exchange_edges_bus2['dest_stop'] = exchange_edges_bus2['dest'].str.split('-').str[2]
exchange_edges_bus2

In [ ]:
na_buses = exchange_edges_bus2[exchange_edges_bus2['wait_time'].isna()]
na_buses['line'] = na_buses['lines'].str.split(' - ').str[1]
na_buses['line'].unique()

In [ ]:
median = exchange_edges_bus2['wait_time'].median()
print(exchange_edges_bus2['wait_time'].mean())
exchange_edges_bus2['wait_time'] = exchange_edges_bus2['wait_time'].fillna(median)
print(exchange_edges_bus2['wait_time'].mean())

# Metro

In [ ]:
exchange_edges_metro = exchange_edges[exchange_edges['dest'].str.startswith('M')]
exchange_edges_metro['lines'].value_counts()

In [ ]:
stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) <= 11]
stop_times

In [ ]:
stops = pd.read_table('Data/GTFS_TMB/stops.txt', sep=',')[['stop_id','stop_name']]
stops

In [ ]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]
stops_w_times

In [ ]:
trips = pd.read_table('Data/GTFS_TMB/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
routes = pd.read_table('Data/GTFS_TMB/routes.txt', sep=',')[['route_id','route_short_name']]
metro = ['L1','L2','L3','L4','L5']
trips = routes.merge(trips, on='route_id', how='left')
trips = trips[trips['route_short_name'].isin(metro)]
trips

In [ ]:
times_w_routes = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['stop_id','stop_name','departure_time','route_id','route_short_name','trip_headsign'], keep='first', inplace=True)
times_w_routes = times_w_routes[times_w_routes['route_id'].notna()]
times_w_routes

In [96]:
times_w_routes[times_w_routes['route_short_name'] == 'L1'].groupby('trip_id')['stop_id'].nunique()

trip_id
1.1.11924033    29
1.1.11924034    30
1.1.11924035    30
1.1.11924036    30
1.1.11924037    30
                ..
1.1.11924383    30
1.1.11924384    30
1.1.11924385    30
1.1.11924386    30
1.1.11924387     4
Name: stop_id, Length: 96, dtype: int64

In [98]:
times_w_routes[times_w_routes['route_short_name'] == 'L1']['stop_id'].nunique()

30

In [ ]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id','route_short_name','trip_headsign', 'departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )

times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_wait_times = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .assign(wait_time=lambda df: df['interval_minutes'] / 2)
    .groupby(['route_id','route_short_name', 'stop_id', 'stop_name','trip_headsign'], as_index=False)['wait_time']
    .mean()
)
avg_wait_times = avg_wait_times.groupby(['route_id','route_short_name', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times

In [ ]:
exchange_edges_metro['dest_name'] = exchange_edges_metro['tram'].str.split(' - ').str[-1]
exchange_edges_metro['dest_name'] = exchange_edges_metro['dest_name'].replace({'Av. de Xile':'Ernest Lluch'})

In [ ]:
exchange_edges_metro = exchange_edges_metro.merge(avg_wait_times[['stop_name','wait_time']], left_on='dest_name', right_on='stop_name', how='left')
exchange_edges_metro = exchange_edges_metro[['origen', 'dest','dest_name','tram', 'mode','lines','type','time',  'wait_time','directed','geometry']]
exchange_edges_metro

In [ ]:
exchange_edges_metro[exchange_edges_metro['wait_time'].isna()]

# Save

In [ ]:
exchanges = pd.concat([exchange_edges_tram, exchange_edges_fgc, exchange_edges_bus, exchange_edges_metro], ignore_index=True)
exchanges.to_csv('Edges/Exchanges_with_Wait_Times.csv', index=False)